 # 02_C1.py : C1, probability seeding (calibrated, per-class precision-targeted thresholds)

 Needs the finished `units_C0` and `fold_splits.json` from 01_C0.py. Z-score preprocessing, stitched evaluation.

 Each round: temperature-calibrate the labeller on the fold's VAL animals, set per-class confidence thresholds for the target

 precision, pseudo-label PS candidate windows from the unlabelled pool, fine-tune FROM THE BASELINE on labelled + pseudo data;

 the student becomes the next labeller. Rows: `pseudo` per round + `gated` (best validated round or baseline).

 Output: `units_C1/fXX_sY.pkl` + `models/C1/*.pth`. Resumable.

In [ ]:
from thesis_common import *
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp

POOLS = ["A"]                 # "A" (controlled comparison, all seeds); add "B", "AB" for the pool factor (seed 0 only)
init_data(unlab=sorted(set("".join(POOLS))))

THR_MIN_PRED, THR_MIN_CORRECT, THR_FALLBACK, THR_DISABLED, THR_FLOOR = 500, 300, .99, 1.01, .60


In [ ]:
@torch.no_grad()
def probs_flat(model, X, bs=128, cap=2_000_000, seed=0):
    model.eval(); P = []
    for s in range(0, len(X), bs):
        xb = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).float().permute(0, 2, 1).to(dev)
        P.append(model(xb).permute(0, 2, 1).reshape(-1, NC).cpu().numpy())
    P = np.concatenate(P)
    return P, np.random.RandomState(seed).permutation(len(P))[:cap]

def fit_T(P, y, n=300_000):
    k = np.random.RandomState(0).permutation(len(P))[:n]; lp = np.log(np.clip(P[k], 1e-8, 1)).astype(np.float64); yy = y[k]
    nll = lambda lt: -(lambda z: (z[np.arange(len(yy)), yy] - logsumexp(z, 1)).mean())(lp / np.exp(lt))
    return float(np.exp(minimize_scalar(nll, bounds=(np.log(.5), np.log(20)), method="bounded").x))

def apply_T(P, T):
    z = np.log(np.clip(P, 1e-8, 1)) / T; return np.exp(z - logsumexp(z, 1, keepdims=True))

def ece(P, y, nb=15):
    conf, ok = P.max(1), P.argmax(1) == y; b = np.clip(np.digitize(conf, np.linspace(0, 1, nb + 1)) - 1, 0, nb - 1); e = 0.0
    for i in range(nb):
        m = b == i
        if m.any(): e += m.mean() * abs(ok[m].mean() - conf[m].mean())
    return float(e)

def class_thresholds(P, y):
    conf, pred = P.max(1), P.argmax(1); thr = {}
    for c, nm in enumerate(CLASS_NAMES):
        tgt = TARGET_PREC.get(nm, TARGET_PREC["default"]); sel = pred == c
        if sel.sum() < THR_MIN_PRED: thr[c] = THR_FALLBACK; continue
        o = np.argsort(-conf[sel]); cs, ok = conf[sel][o], (y[sel] == c)[o]; nc = np.cumsum(ok); prec = nc / np.arange(1, len(ok) + 1)
        good = (prec >= tgt) & (nc >= THR_MIN_CORRECT)
        thr[c] = THR_DISABLED if not good.any() else float(max(cs[np.where(good)[0].max()], THR_FLOOR))
    return thr

def calibrate(model, E_val):
    P, _ = probs_flat(model, E_val["X"]); y = E_val["Y"].ravel(); T = fit_T(P, y); Pc = apply_T(P, T)
    return T, class_thresholds(Pc, y), {"T": T, "ece_before": ece(P, y), "ece_after": ece(Pc, y)}

@torch.no_grad()
def prob_labels(model, X, T, thr, bs=128):
    model.eval(); th = torch.tensor([thr[c] for c in range(NC)], dtype=torch.float32, device=dev); Ys, Ms = [], []
    for s in range(0, len(X), bs):
        xb = torch.from_numpy(np.ascontiguousarray(X[s:s + bs])).float().permute(0, 2, 1).to(dev)
        p = F.softmax(torch.log(model(xb).clamp_min(1e-8)) / T, 1); conf, pred = p.max(1)
        Ys.append(pred.cpu().numpy().astype(np.int8)); Ms.append((conf >= th[pred]).cpu().numpy())
    Y, M = np.concatenate(Ys), np.concatenate(Ms); return Y, M, M.sum(1) >= MIN_CONF


In [ ]:
def run_c1(pool, ctx):
    fi, seed = ctx["fi"], ctx["seed"]; rows, qrows, srows, crows = [], [], [], []
    rng = np.random.RandomState(seed * 7 + fi); ent = make_pool(pool); Xc, srcc = draw_windows(ent, N_CAND, rng)
    labeller = ctx["base"]
    base_meta = dict(arm=f"C1|{pool}|natural|pf1.0", cond="C1", pool=pool, sampling="natural", pool_frac=1.0, exp="pool",
                     fold=fi + 1, test_ind=ctx["test_ind"], seed=seed, frac=1.0)
    for r in range(1, PS_ITERS + 1):
        meta = {**base_meta, "round": r}
        T, thr, cal = calibrate(labeller, ctx["EV"]); crows.append({**meta, **cal, **{f"thr_{CLASS_NAMES[c]}": thr[c] for c in range(NC)}})
        lab = lambda X: prob_labels(labeller, X, T, thr)
        for nm, E in (("VAL", ctx["EV"]), ("TEST", ctx["ET"])):
            Y, M, keep = lab(E["X"]); qrows += quality_rows(E["Y"], Y, M, keep, {**meta, "eval_set": nm})
        P, _ = probs_flat(labeller, ctx["ET"]["X"]); yt = ctx["ET"]["Y"].ravel()
        crows[-1].update(test_ece_raw=ece(P, yt), test_ece_cal=ece(apply_T(P, T), yt))
        Yc, Mc, keepc = lab(Xc); sel, Ms, info = select_pseudo(Yc, Mc, keepc, srcc, SEED + fi * 100 + r)
        srows.append({**meta, **info, "n_candidates": len(Xc), "n_keep": int(keepc.sum())})
        if len(sel) == 0: print("no pseudo windows; stopping chain"); break
        ps = PseudoSet(Xc[sel], Yc[sel].astype(np.int64), Ms)
        student = finetune(ctx["base"], ctx["names_train"], 5000 + fi * 100 + r + seed * 7, pseudo=ps)
        save_state(student, "C1", fi, seed, f"{pool}_r{r}")
        row = row_of({**meta, "kind": "pseudo"}, student, ctx); rows.append(row)
        print(f"C1 {pool} round {r}: test macro {row['test_macro']:.3f} | test target {row['test_target']:.3f}")
        labeller = student; del ps
    return finish_chain(rows, ctx, base_meta), qrows, srows, crows


In [ ]:
run_stage("C1", run_c1, POOLS)
print("C1 units finished:", len(list((OUT / "units_C1").glob("*.pkl"))))
